In [19]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Goldfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source": "fish-pets-doc"},
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source": "bird-pets-doc"},
    ),
    Document(
        page_content="Rabbits are social animals that need plenty of space to hop around.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

In [20]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key=os.getenv("GROQ_API_KEY")

os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

llm=ChatGroq(groq_api_key=groq_api_key, model="qwen/qwen3-32b")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013F0E70BA00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013F0E6F8730>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [21]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 149.49it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [22]:
## VectorStores
from langchain_chroma import Chroma

vectorstore=Chroma.from_documents(documents,embedding=embeddings)
vectorstore


In [23]:
vectorstore.similarity_search("cat")

[Document(id='820c5f49-3984-46c9-b42f-316b87e36044', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='353ebc86-9964-4904-b1bc-3256ce989890', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='93f57f07-d974-45bd-80a6-9a961aaf0675', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='641a177f-ef9c-44cd-b92c-699864b4b26d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

In [24]:
await vectorstore.asimilarity_search("cat")

[Document(id='820c5f49-3984-46c9-b42f-316b87e36044', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='353ebc86-9964-4904-b1bc-3256ce989890', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='93f57f07-d974-45bd-80a6-9a961aaf0675', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.'),
 Document(id='641a177f-ef9c-44cd-b92c-699864b4b26d', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]

## Retrievers

In [25]:
retriever=vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={"k":1}
)
retriever.batch(["cat",'dog'])

[[Document(id='93f57f07-d974-45bd-80a6-9a961aaf0675', metadata={'source': 'mammal-pets-doc'}, page_content='Cats are independent pets that often enjoy their own space.')],
 [Document(id='10bc4476-3eef-4cff-899d-a976f8924e63', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companions, known for their loyalty and friendliness.')]]

In [27]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message=''' 
answer the following question from the following context.
{question}

context:{context}
'''
prompt=ChatPromptTemplate.from_messages(["human",message])
rag_chain={'context':retriever,'question':RunnablePassthrough()}|prompt|llm
response=rag_chain.invoke('tell me about dogs')
response.content

'<think>\nOkay, the user asked "tell me about dogs" and provided a context document. Let me check the context first. The document mentions that dogs are great companions, loyal, and friendly. The source is \'mammal-pets-doc\', so probably a general info text.\n\nI need to answer based solely on the given context. The user probably wants a brief answer highlighting the key points from the document. Since there\'s only one sentence in the context, I should paraphrase that. Maybe mention their companionship, loyalty, and friendliness. No need to add extra info beyond what\'s provided. Make sure the answer is concise and directly uses the context. Also, since the user is a human, keep the tone friendly and straightforward. Let me put that together.\n</think>\n\nDogs are highly valued as companions due to their loyalty and friendliness. They are known for forming strong bonds with humans and are often kept as pets for their affectionate and devoted nature.'